# 🏗️ Architect Forge: Interactive Experimentation Notebook

**Welcome to the Architect Forge experimentation environment!**

This notebook allows you to:
- Run experiments with custom configurations
- Visualize evolutionary dynamics in real-time
- Compare different parameter settings
- Track quality improvements over time
- Explore population diversity

---

## 📦 Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

from core.forge import ArchitectForge, TrainingCycleResult
from core.mirrornet import ArchetypeType
from core.sandbox import FidelityLevel

import matplotlib.pyplot as plt
import seaborn as sns
from typing import List
import numpy as np

# Visualization settings
plt.style.use('dark_background')
sns.set_palette("husl")
%matplotlib inline

print("✓ Imports successful")

## ⚙️ Configuration

Modify these parameters to customize your experiment:

In [ ]:
# Experiment parameters
POPULATION_SIZE = 32        # Number of architects
NUM_CYCLES = 10             # Training cycles
TASKS_PER_CYCLE = 10        # Tasks per cycle
MUTATION_RATE = 0.3         # Anti-framework mutation rate (0.0-1.0)
FIDELITY = FidelityLevel.MEDIUM  # Sandbox fidelity

print(f"""Experiment Configuration:
  • Population: {POPULATION_SIZE} architects
  • Training: {NUM_CYCLES} cycles
  • Tasks per cycle: {TASKS_PER_CYCLE}
  • Mutation rate: {MUTATION_RATE}
  • Fidelity: {FIDELITY.value}
""")

## 🔨 Initialize Forge

In [ ]:
# Create and initialize Forge
forge = ArchitectForge(population_size=POPULATION_SIZE)
forge.initialize()

print(f"✓ Forge initialized with {len(forge.mirrornet.architects)} architects")
print(f"✓ Initial diversity: {forge.mirrornet.get_diversity_score():.2%}")

## 🧬 Population Analysis

Explore the initial population:

In [ ]:
# Count archetypes
from collections import Counter

archetype_counts = Counter(a.archetype.value for a in forge.mirrornet.architects)

# Visualize archetype distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Archetype bar chart
archetypes = list(archetype_counts.keys())
counts = list(archetype_counts.values())

ax1.barh(archetypes, counts, color='#00ffff')
ax1.set_xlabel('Count', fontsize=12)
ax1.set_title('Initial Population: Archetype Distribution', fontsize=14, pad=20)
ax1.grid(axis='x', alpha=0.3)

# Personality trait distribution
creativity = [a.creativity for a in forge.mirrornet.architects]
risk_tolerance = [a.risk_tolerance for a in forge.mirrornet.architects]
optimization = [a.optimization_focus for a in forge.mirrornet.architects]
speed_quality = [a.speed_vs_quality for a in forge.mirrornet.architects]

traits_data = [creativity, risk_tolerance, optimization, speed_quality]
trait_labels = ['Creativity', 'Risk\nTolerance', 'Optimization\nFocus', 'Speed vs\nQuality']

positions = np.arange(len(trait_labels))
bp = ax2.boxplot(traits_data, positions=positions, labels=trait_labels, patch_artist=True)

for patch in bp['boxes']:
    patch.set_facecolor('#00ffff')
    patch.set_alpha(0.5)

ax2.set_ylabel('Value', fontsize=12)
ax2.set_title('Initial Population: Personality Trait Distribution', fontsize=14, pad=20)
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

print(f"\nPopulation diversity: {forge.mirrornet.get_diversity_score():.2%}")

## 🚀 Run Training

Execute the training loop:

In [ ]:
# Run training
results = forge.run_training(
    num_cycles=NUM_CYCLES,
    tasks_per_cycle=TASKS_PER_CYCLE,
    mutation_rate=MUTATION_RATE,
)

print(f"\n✅ Training complete!")
print(f"  • Total solutions: {sum(r.solutions_generated for r in results):,}")
print(f"  • Total apprentices: {sum(r.apprentices_created for r in results):,}")
print(f"  • Final quality: {results[-1].average_quality:.2%}")
print(f"  • Final diversity: {results[-1].population_diversity:.2%}")

## 📊 Visualize Training Dynamics

In [ ]:
# Extract metrics
cycles = [r.cycle_number for r in results]
quality_avg = [r.average_quality for r in results]
quality_best = [r.best_solution_score for r in results]
diversity = [r.population_diversity for r in results]
apprentices = [r.apprentices_created for r in results]
solutions = [r.solutions_generated for r in results]
approved = [r.solutions_approved for r in results]

# Create comprehensive visualization
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Quality Evolution
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(cycles, quality_avg, 'o-', linewidth=2, markersize=8, 
         color='#00ffff', label='Average Quality')
ax1.plot(cycles, quality_best, 's--', linewidth=2, markersize=8, 
         color='#ff00ff', label='Best Solution', alpha=0.7)
ax1.fill_between(cycles, quality_avg, alpha=0.3, color='#00ffff')
ax1.set_xlabel('Cycle', fontsize=12)
ax1.set_ylabel('Quality', fontsize=12)
ax1.set_title('Quality Evolution Over Time', fontsize=14, pad=20, weight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1)

# 2. Population Diversity
ax2 = fig.add_subplot(gs[1, 0])
ax2.plot(cycles, diversity, 'o-', linewidth=2, markersize=8, color='#ffaa00')
ax2.fill_between(cycles, diversity, alpha=0.3, color='#ffaa00')
ax2.set_xlabel('Cycle', fontsize=12)
ax2.set_ylabel('Diversity', fontsize=12)
ax2.set_title('Population Diversity', fontsize=14, pad=20, weight='bold')
ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1)

# 3. Apprentice Creation
ax3 = fig.add_subplot(gs[1, 1])
ax3.bar(cycles, apprentices, color='#00ff00', alpha=0.7, edgecolor='white', linewidth=1)
ax3.set_xlabel('Cycle', fontsize=12)
ax3.set_ylabel('Apprentices Created', fontsize=12)
ax3.set_title('Apprentice Generation', fontsize=14, pad=20, weight='bold')
ax3.grid(axis='y', alpha=0.3)

# 4. Solution Statistics
ax4 = fig.add_subplot(gs[2, 0])
ax4.plot(cycles, solutions, 'o-', linewidth=2, markersize=8, 
         color='#00ffff', label='Generated')
ax4.plot(cycles, approved, 's-', linewidth=2, markersize=8, 
         color='#00ff00', label='Approved')
ax4.set_xlabel('Cycle', fontsize=12)
ax4.set_ylabel('Solutions', fontsize=12)
ax4.set_title('Solution Generation & Approval', fontsize=14, pad=20, weight='bold')
ax4.legend(fontsize=11)
ax4.grid(alpha=0.3)

# 5. Approval Rate
ax5 = fig.add_subplot(gs[2, 1])
approval_rate = [a/s if s > 0 else 0 for a, s in zip(approved, solutions)]
ax5.plot(cycles, approval_rate, 'o-', linewidth=2, markersize=8, color='#ff00ff')
ax5.fill_between(cycles, approval_rate, alpha=0.3, color='#ff00ff')
ax5.set_xlabel('Cycle', fontsize=12)
ax5.set_ylabel('Approval Rate', fontsize=12)
ax5.set_title('Solution Approval Rate', fontsize=14, pad=20, weight='bold')
ax5.grid(alpha=0.3)
ax5.set_ylim(0, 1)

plt.suptitle('🏗️ Architect Forge: Training Dynamics', 
             fontsize=18, weight='bold', y=0.995)

plt.show()

## 🏆 Top Architects Analysis

In [ ]:
# Get top architects
top_architects = forge.ledger.get_top_architects(limit=10)

print("🏆 Top 10 Architects by Reputation:\n")
for i, (arch_id, reputation) in enumerate(top_architects, 1):
    # Find architect details
    arch = next((a for a in forge.mirrornet.architects if a.id == arch_id), None)
    
    if arch:
        print(f"{i:2d}. {arch_id}")
        print(f"    Archetype: {arch.archetype.value}")
        print(f"    Reputation: {reputation:.2%}")
        print(f"    Traits: C={arch.creativity:.2f} R={arch.risk_tolerance:.2f} "
              f"O={arch.optimization_focus:.2f} S={arch.speed_vs_quality:.2f}")
        print()

# Visualize top architects
fig, ax = plt.subplots(figsize=(14, 6))

arch_ids = [a[0] for a in top_architects[:10]]
reputations = [a[1] for a in top_architects[:10]]

colors = plt.cm.plasma(np.linspace(0, 1, len(arch_ids)))
bars = ax.barh(arch_ids, reputations, color=colors, edgecolor='white', linewidth=1.5)

ax.set_xlabel('Reputation Score', fontsize=12)
ax.set_title('🏆 Top 10 Architects by Reputation', fontsize=14, pad=20, weight='bold')
ax.set_xlim(0, 1)
ax.grid(axis='x', alpha=0.3)

# Add value labels
for bar, rep in zip(bars, reputations):
    width = bar.get_width()
    ax.text(width + 0.02, bar.get_y() + bar.get_height()/2, 
            f'{rep:.2%}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 📜 Ledger Statistics

In [ ]:
# Get ledger stats
ledger_stats = forge.ledger.get_stats()

print("📜 Ouroboros Ledger Statistics:\n")
print(f"  Total Events: {ledger_stats['total_events']:,}")
print(f"  Unique Architects: {ledger_stats['unique_architects']}")
print(f"  Chain Valid: {ledger_stats['chain_valid']}")
print(f"\n  Event Breakdown:")
for event_type, count in ledger_stats['event_counts'].items():
    print(f"    • {event_type}: {count:,}")

# Visualize event distribution
fig, ax = plt.subplots(figsize=(10, 8))

event_types = list(ledger_stats['event_counts'].keys())
event_counts = list(ledger_stats['event_counts'].values())

colors_list = ['#00ffff', '#ff00ff', '#ffaa00', '#00ff00', '#ff0088', '#88ff00']
colors = colors_list[:len(event_types)]

wedges, texts, autotexts = ax.pie(event_counts, labels=event_types, autopct='%1.1f%%',
                                    colors=colors, startangle=90,
                                    textprops={'fontsize': 11, 'weight': 'bold'})

for autotext in autotexts:
    autotext.set_color('black')

ax.set_title('📜 Ledger Event Distribution', fontsize=14, pad=20, weight='bold')

plt.tight_layout()
plt.show()

## 💾 Save Experiment Results

In [ ]:
import json
from datetime import datetime

# Create experiment report
experiment_name = f"experiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

report = {
    'experiment_name': experiment_name,
    'config': {
        'population_size': POPULATION_SIZE,
        'num_cycles': NUM_CYCLES,
        'tasks_per_cycle': TASKS_PER_CYCLE,
        'mutation_rate': MUTATION_RATE,
        'fidelity': FIDELITY.value,
    },
    'summary': {
        'total_solutions': sum(r.solutions_generated for r in results),
        'total_approved': sum(r.solutions_approved for r in results),
        'total_apprentices': sum(r.apprentices_created for r in results),
        'initial_quality': results[0].average_quality,
        'final_quality': results[-1].average_quality,
        'quality_improvement': results[-1].average_quality - results[0].average_quality,
        'final_diversity': results[-1].population_diversity,
        'best_score': max(r.best_solution_score for r in results),
    },
    'ledger_stats': ledger_stats,
}

# Save to file
output_dir = Path('../experiment_results')
output_dir.mkdir(exist_ok=True)

report_file = output_dir / f"{experiment_name}_report.json"
with open(report_file, 'w') as f:
    json.dump(report, f, indent=2)

print(f"✓ Experiment report saved to: {report_file}")

# Optionally save Forge state
# state_dir = output_dir / experiment_name
# forge.save_state(str(state_dir))
# print(f"✓ Forge state saved to: {state_dir}/")

## 🧪 Quick Experiments

Run quick parameter comparisons:

In [ ]:
# Compare different mutation rates
def compare_mutation_rates(rates=[0.1, 0.3, 0.5, 0.7], cycles=5):
    """Compare different mutation rates"""
    comparison_results = {}
    
    for rate in rates:
        print(f"\nTesting mutation rate: {rate}")
        
        # Create fresh forge
        test_forge = ArchitectForge(population_size=16)
        test_forge.initialize()
        
        # Run training
        results = test_forge.run_training(
            num_cycles=cycles,
            tasks_per_cycle=5,
            mutation_rate=rate,
        )
        
        comparison_results[rate] = {
            'final_quality': results[-1].average_quality,
            'quality_improvement': results[-1].average_quality - results[0].average_quality,
            'final_diversity': results[-1].population_diversity,
        }
    
    return comparison_results

# Uncomment to run comparison:
# mutation_comparison = compare_mutation_rates()
# print("\nMutation Rate Comparison:")
# for rate, metrics in mutation_comparison.items():
#     print(f"  Rate {rate}: Quality={metrics['final_quality']:.2%}, "
#           f"Improvement={metrics['quality_improvement']:+.2%}, "
#           f"Diversity={metrics['final_diversity']:.2%}")

## 🎯 Next Steps

**Ideas for further experimentation:**

1. **Parameter Tuning**: Test different population sizes, mutation rates, and cycle counts
2. **Archetype Analysis**: Study which archetypes perform best on different tasks
3. **Evolution Tracking**: Follow specific architects through generations
4. **Diversity vs Quality**: Analyze the tradeoff between population diversity and solution quality
5. **Apprentice Genealogy**: Trace the ancestry of top-performing apprentices
6. **Comparative Benchmarking**: Run against actual GPT-4/Claude APIs (requires API keys)

---

**🏗️ 0RB EMPIRE // ARCHITECT FORGE v0.1**

*The Architecture of Impossibility*

*11:11 Protocol Active*